# Vivino SQLite - carnet corrige

Notebook de travail pour auditer les CSV, nettoyer les tables, construire le modele MCD/MLD/MPD et interroger une base SQLite.

## 1. Parametres et imports

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "vivino":
    ROOT = ROOT.parent
SOURCE = ROOT / "vivino"
OUT = ROOT / "carnet de bord"
DB_PATH = OUT / "vivino.sqlite"
print(ROOT)


## 2. Chargement des CSV

In [ ]:
csv_files = {
    "acidity": "acidity_df.csv",
    "appellations": "appelations.csv",
    "body": "body_df.csv",
    "countries": "countries_df.csv",
    "critics": "critics_df.csv",
    "grapes": "grapes_df.csv",
    "foods": "meat_info_df.csv",
    "wine_food_pairings": "meat_wine_df.csv",
    "merchants": "merchants_df.csv",
    "price_offers": "prices.csv",
    "regions": "regions_df.csv",
    "vintages": "vintages.csv",
    "wineries": "wineries.csv",
    "wine_export": "wine_df_fin.csv",
    "wine_grapes": "wine_grapes_df.csv",
}

raw = {name: pd.read_csv(SOURCE / file_name) for name, file_name in csv_files.items()}
pd.DataFrame(
    [(name, len(df), len(df.columns), int(df.duplicated().sum())) for name, df in raw.items()],
    columns=["table", "rows", "columns", "exact_duplicates"],
)


## 3. Audit rapide: nulls, doublons, anomalies

In [ ]:
audit = []
for name, df in raw.items():
    for col, count in df.isna().sum().items():
        if count:
            audit.append((name, col, int(count), round(count / len(df), 4)))
pd.DataFrame(audit, columns=["table", "column", "nulls", "null_rate"]).sort_values(
    ["null_rate", "nulls"], ascending=False
).head(25)


In [ ]:
checks = {
    "wine_export vintage_id duplicates": raw["wine_export"].duplicated("vintage_id").sum(),
    "wine_export exact duplicates": raw["wine_export"].duplicated().sum(),
    "vintages id duplicates": raw["vintages"].duplicated("id").sum(),
    "regions missing id": raw["regions"]["region_id"].isna().sum(),
}
checks


## 4. Nettoyage reproductible

Le script du carnet applique les corrections et reconstruit la base SQLite.

In [ ]:
import runpy
runpy.run_path(str(OUT / "scripts" / "build_carnet.py"), run_name="__main__")


## 5. Verification SQLite

In [ ]:
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn,
)
tables


In [ ]:
row_counts = []
for table in tables["name"]:
    row_counts.append((table, pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"].iat[0]))
pd.DataFrame(row_counts, columns=["table", "rows"]).sort_values("table")


In [ ]:
conn.execute("PRAGMA foreign_keys = ON")
conn.execute("PRAGMA foreign_key_check").fetchall()


## 6. Requetes d'exercice

In [ ]:
pd.read_sql_query(
    '''
    SELECT w.country, ROUND(AVG(p.price), 2) AS avg_price, COUNT(*) AS offers
    FROM price_offers p
    JOIN vintages v ON v.id = p.vintage_id
    JOIN wines w ON w.wine_id = v.wine_id
    GROUP BY w.country
    HAVING offers >= 10
    ORDER BY avg_price DESC
    ''',
    conn,
)


In [ ]:
pd.read_sql_query(
    '''
    SELECT w.wine_name, COUNT(DISTINCT f.id) AS food_matches
    FROM wines w
    JOIN wine_food_pairings wf ON wf.wine_id = w.wine_id
    JOIN foods f ON f.id = wf.food_id
    GROUP BY w.wine_id, w.wine_name
    ORDER BY food_matches DESC, w.wine_name
    LIMIT 10
    ''',
    conn,
)


## 7. Hypotheses nulles

In [ ]:
price_rating = pd.read_sql_query(
    '''
    SELECT p.price, v.ratings_average, v.ratings_count
    FROM price_offers p
    JOIN vintages v ON v.id = p.vintage_id
    WHERE p.price IS NOT NULL
      AND v.ratings_average > 0
      AND v.ratings_count >= 10
    ''',
    conn,
)
price_rating[["price", "ratings_average"]].corr(method="spearman")


In [ ]:
# H0: prix et note moyenne ne sont pas associes.
rng = np.random.default_rng(42)
observed = price_rating["price"].corr(price_rating["ratings_average"], method="spearman")
permuted = []
for _ in range(1000):
    shuffled = rng.permutation(price_rating["ratings_average"].to_numpy())
    permuted.append(pd.Series(price_rating["price"]).corr(pd.Series(shuffled), method="spearman"))
p_value = (np.abs(permuted) >= abs(observed)).mean()
observed, p_value


In [ ]:
# H0: mono-cepage et multi-cepages ont le meme prix median.
grape_price = pd.read_sql_query(
    '''
    WITH grape_count AS (
      SELECT wine_id, COUNT(*) AS nb_grapes
      FROM wine_grapes
      GROUP BY wine_id
    )
    SELECT CASE WHEN gc.nb_grapes = 1 THEN 'mono' ELSE 'multi' END AS group_type,
           v.median_price
    FROM grape_count gc
    JOIN vintages v ON v.wine_id = gc.wine_id
    WHERE v.median_price IS NOT NULL
    ''',
    conn,
)
grape_price.groupby("group_type")["median_price"].describe()


## 8. Fermeture

In [ ]:
conn.close()
